In [ ]:
import json
import urllib.request

# URL to fetch the up-to-date master database
MASTER_DATA_URL = "https://chunithm.beerpsi.cc/songs"

# The name of your local file containing the target songs with IDs
INPUT_FILE = "paradiselost_offline_songs.json" 
OUTPUT_FILE = "paradiselost_offline_songs.json"

# Load the target data from your attached file
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    target_data = json.load(f)

# Extract all unique IDs from the target file
target_ids = {song['id'] for song in target_data if 'id' in song}

# Fetch master data dynamically over the network
print("Fetching chunirec master song list")
req = urllib.request.Request(MASTER_DATA_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    master_data = json.loads(response.read().decode('utf-8'))

filtered_songs = []
matched_ids = set()
unmatched_ids = target_ids.copy()

for song in master_data:
    song_id = song.get('id')
    
    # Match directly by ID
    if song_id in target_ids:
        
        # Create a working copy of the song to filter out ULT difficulties
        song_to_add = song.copy()
        if 'charts' in song_to_add:
            song_to_add['charts'] = [c for c in song_to_add['charts'] if c.get('difficulty') != 'ULT']
            
        filtered_songs.append(song_to_add)
        matched_ids.add(song_id)
        unmatched_ids.discard(song_id)

# Export the final filtered list
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(filtered_songs, f, ensure_ascii=False, indent=2)

print(f"Process complete! Total matched: {len(filtered_songs)} / {len(target_ids)}")

if unmatched_ids:
    print("\nStill unmatched (IDs missing from the master list):")
    for missing_id in unmatched_ids:
        print(f" - ID: {missing_id}")

Fetching master song list dynamically...
Process complete! Total matched: 275 / 275
